# Settling the double — the learning-rate schedule (and why it isn't enough)

Notebook 02 traced the naive plateau to an oscillating value on the terminal `double`. The obvious
fix is to **decay the learning rate** so the estimate stops bouncing and converges. This notebook
chases that fix all the way down, and lands three findings — the third is the important one:

1. **The lever is a low learning rate, not the schedule *shape*.** Harmonic, linear and constant all
   fall on one curve: the double's settling is governed purely by the lr *during the back half*.
2. **There's a reach-vs-settle frontier.** A single global schedule trades one for the other —
   settle the double and you under-learn; learn fully and it won't settle.
3. **Settling the double does *not* close the agreement gap.** The tightest-settled run still trails
   linear. So the oscillation was a *symptom we could fix*, not the thing capping agreement — which
   sends us to notebooks 04–05 for the real cause.

In [ ]:
import sys; sys.path.insert(0, '.')
import json, math, statistics as st
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from blackjack_rl.analysis_loader import load_runs

df = load_runs(); dqn = df[df.method == 'dqn']

def lr_at_fn(row):
    """Reconstruct a run's lr(episode) from its config — mirrors blackjack_rl/schedules.py."""
    sch, s, e, hold, n = row.lr_schedule, row.lr, row.lr_end, row.lr_hold_until, row.episodes
    last = max(1, n - 1)
    if sch == 'constant' or s is None:
        return lambda i: s
    def shape(p):
        if sch == 'linear': return s + (e - s) * p
        if sch == 'exponential': return s * (e / s) ** p
        return s / (1 + (s / e - 1) * p)            # harmonic
    if not hold:
        return lambda i: shape(i / last)
    span = max(1, last - hold)
    return lambda i: s if i < hold else shape((i - hold) / span)

def backhalf_lr(row):
    """Average lr over the back half of training — the window where double-std is measured."""
    if row.lr is None: return None
    f = lr_at_fn(row); last = max(1, row.episodes - 1)
    return st.mean(f(int(last * x)) for x in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0))

def agreement_curve(path):
    lc = json.load(open(Path(path) / 'record.json', encoding='utf-8'))['learning_curve']
    return [(cp['episode'], cp['agreement']) for cp in lc if cp.get('agreement') is not None]

## The lever is low lr, not the schedule shape

The double's oscillation is *constant-gain variance* (02): the estimate's steady-state wobble scales
with the step size. So the prediction is that the back-half **double-std** depends only on the lr in
that window — regardless of whether the schedule is constant, linear, or harmonic. Plotting every
run that way, they collapse onto a single trend: harmonic settles best **only because its back-half
lr is lowest** (the 1/k tail), not because the shape is special.

In [ ]:
sub = dqn[dqn.soft20_double_std.notna() & dqn.lr.notna()].copy()
sub['bh_lr'] = [backhalf_lr(r) for _, r in sub.iterrows()]
colors = {'constant': '#888888', 'linear': '#378ADD', 'harmonic': '#D85A30', 'exponential': '#7F77DD'}
fig, ax = plt.subplots(figsize=(8, 5))
for sch, g in sub.groupby('lr_schedule'):
    ax.scatter(g.bh_lr, g.soft20_double_std, s=34, c=colors.get(sch, '#888'), label=sch, alpha=0.8, edgecolor='none')
ftd = sub[sub.lr_hold_until > 0]
ax.scatter(ftd.bh_lr, ftd.soft20_double_std, s=120, facecolor='none', edgecolor='#1b3a5b', lw=1.5, label='flat-then-decay')
ax.set_xscale('log'); ax.set_xlabel('back-half average learning rate (log)')
ax.set_ylabel('double-std (back-half oscillation)')
ax.set_title('Settling is set by the lr in the window — not the schedule name'); ax.legend()
plt.tight_layout(); plt.show()
print('median double-std by back-half lr band:')
for lo, hi, name in [(0, 5e-5, 'low ~1e-5 (harmonic tail)'), (5e-5, 4e-4, 'mid ~2e-4 (linear)'), (4e-4, 1, 'high ~1e-3 (constant)')]:
    band = sub[(sub.bh_lr >= lo) & (sub.bh_lr < hi)]
    if len(band): print(f'  {name:28}: median {band.soft20_double_std.median():.2f}  (n={len(band)})')

## The reach-vs-settle frontier

Low lr settles the double — but a *globally* low lr also does too little total learning (it crawls).
So a single global schedule is stuck on a frontier: push lr down to settle and agreement falls; keep
lr up to learn and the double won't settle. The end-sweep traces it directly.

In [ ]:
rows = []
for label, sel in [
    ('harmonic ->1e-5 (settle)', dict(lr_schedule='harmonic', lr_end=1e-5, episodes=1_500_000, lr_hold_until=0)),
    ('harmonic ->1e-4',          dict(lr_schedule='harmonic', lr_end=1e-4, episodes=1_500_000, lr_hold_until=0)),
    ('linear ->0 (reach)',       dict(lr_schedule='linear', episodes=1_500_000, batch=128, reward_baseline='stand')),
]:
    g = dqn
    for k, v in sel.items(): g = g[g[k] == v]
    g = g[g.encoding == 'scalar']
    if len(g): rows.append((label, g.agreement.mean(), g.edge_pct.mean(), g.soft20_double_std.mean()))
print(f"{'schedule':26}{'agree':>8}{'edge':>7}{'dbl-std':>9}")
for lab, a, e, s in rows: print(f'{lab:26}{a:>7.1%}{e:>6.2f}%{s:>9.2f}')
print('\n-> settle (low std) and reach (high agreement) pull in opposite directions.')

## Flat-then-decay: co-designing the schedule with the curriculum

If the frontier comes from forcing *one* lr on the whole run, the escape is to separate the two needs
in time. Hold the lr high while only hit/stand exists (double masked, `--double-after`), then decay
once double is introduced (`--lr-hold-until`) so it settles. This gives the **best settle in the
field and the most stable trajectory** — the design works.

In [ ]:
ftd = dqn[(dqn.lr_hold_until > 0) & (dqn.encoding == 'scalar') & (dqn.episodes == 1_500_000)]
linear = dqn[(dqn.lr_schedule == 'linear') & (dqn.encoding == 'scalar') & (dqn.episodes == 1_500_000)
             & (dqn.batch == 128) & (dqn.reward_baseline == 'stand')]
harm = dqn[(dqn.lr_schedule == 'harmonic') & (dqn.lr_end == 1e-5) & (dqn.episodes == 1_500_000) & (dqn.lr_hold_until == 0)]
print(f"{'config':22}{'agree':>8}{'edge':>7}{'dbl-std':>9}")
for lab, g in [('flat-then-decay', ftd), ('global harmonic ->1e-5', harm), ('linear ->0', linear)]:
    if len(g): print(f'{lab:22}{g.agreement.mean():>7.1%}{g.edge_pct.mean():>6.2f}%{g.soft20_double_std.mean():>9.2f}')

## The dynamics, not just the endpoints

Every number above is a *final* snapshot of a value that's usually still moving — so the endpoint bars
hide the story. Overlaying agreement *over training* makes the frontier legible: constant oscillates
and plateaus low; linear climbs highest but stays noisy to the very end; global harmonic settles
smoothly but caps low; flat-then-decay sits flat through the masked hit/stand phase, then climbs
cleanly once double enters at 500k. The history shows *why* each lands where it does in a way the bar
charts can't.

In [ ]:
curves = [
    ('constant (oscillates, plateaus low)', dqn[(dqn.lr_schedule=='constant') & (dqn.encoding=='scalar') & (dqn.episodes==1_500_000)].iloc[0], '#9AA0A6'),
    ('linear ->0 (reaches high, stays noisy)', dqn[(dqn.lr_schedule=='linear') & (dqn.encoding=='scalar') & (dqn.episodes==1_500_000) & (dqn.batch==128) & (dqn.reward_baseline=='stand')].iloc[0], '#378ADD'),
    ('harmonic ->1e-5 (settles, under-learns)', dqn[(dqn.lr_schedule=='harmonic') & (dqn.lr_end==1e-5) & (dqn.episodes==1_500_000) & (dqn.lr_hold_until==0)].iloc[0], '#D85A30'),
    ('flat-then-decay (switch @500k)', dqn[(dqn.lr_hold_until>0) & (dqn.encoding=='scalar') & (dqn.episodes==1_500_000)].iloc[0], '#534AB7'),
]
fig, ax = plt.subplots(figsize=(10, 5))
for label, row, col in curves:
    c = agreement_curve(row.path)
    ax.plot([e for e, _ in c], [a for _, a in c], color=col, lw=1.3, alpha=0.9, label=label)
ax.axvline(500_000, ls='--', color='#999', lw=1)
ax.text(505_000, 0.70, 'double introduced\n(flat-then-decay)', fontsize=8, color='#666')
ax.set_xlabel('training episode'); ax.set_ylabel('agreement with basic strategy')
ax.set_title('Agreement over training — the dynamics the endpoint bars hide')
ax.legend(fontsize=8, loc='lower right'); plt.tight_layout(); plt.show()
# why we report a back-half mean, not the last checkpoint
fr = curves[-1][1]; cc = agreement_curve(fr.path)
bh = [a for e, a in cc if e > fr.episodes // 2]
print(f'flat-then-decay: final snapshot {cc[-1][1]:.3f}  vs  back-half mean {np.mean(bh):.3f} (+-{np.std(bh):.3f})')
print('the last checkpoint is a random phase of the wobble -> report the back-half mean, not the final point.')

## The twist: settling the double does *not* close the gap

Here is the result that redirects the whole investigation. Flat-then-decay settles the double tightest
of anything (std ~0.12) — and still **trails linear on both agreement and edge**. We beat the
oscillation completely and the agreement barely moved. So the oscillating double was a *symptom* we
could cure, but it was never what capped agreement. The cap is something the learning-rate axis
cannot touch — which is exactly what notebooks 04 (rule out the remaining training fixes) and 05
(the real cause: representation) take up.

Mechanistically the contrast is clean: settling is a *variance* fix (stop the wobble), but the
residual error turns out to be a *bias* (a systematically wrong action in boundary cells), and no
amount of variance reduction moves a bias.